# Trend-only baseline: OLS + IRLS, no LOWESS

Fits `F0trend` using `nonlinear_fit` (OLS pre-pass + IRLS) with:
- `fixed_sigma=None` — auto-computes sigma from MAD(residuals) each IRLS iteration (fixes the flipping bug from `first_try`)
- No LOWESS fluctuation step — `F0trend` is used directly as the baseline

**Output:** `/scratch/trend_only_default/<session_key>/`  
**Base data (F, dff, baselines, sczdrift):** loaded from `/scratch/first_try/<session_key>/`

Two path options below — run Option B (from `/scratch/first_try/`) only.

In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import skew
import jax
import jax.numpy as jnp
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from aind_ophys_utils.signal_utils import noise_std
from baseline_fitting import AsymmetricTukeyBiweight, nonlinear_fit

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.close_figures = True

In [3]:
def model(params, t, xp=np):
    """Biphasic bleaching + saturating brightening trend model.
    params: [b_inf, b_slow, b_fast, b_bright, t_slow, t_fast, t_bright]
    """
    b_inf, b_slow, b_fast, b_bright, t_slow, t_fast, t_bright = params
    E_slow = xp.exp(-t / t_slow)
    E_fast = xp.exp(-t / t_fast)
    E_bright = xp.exp(-t / t_bright)
    return b_inf + b_slow * E_slow + b_fast * E_fast - b_bright * E_bright

M = AsymmetricTukeyBiweight(c_pos=2, c_neg=3)

save_dir_base = Path('/root/capsule/scratch/trend_only_default')
save_dir_base.mkdir(parents=True, exist_ok=True)

---
## Option A — Load from `/data/` (requires live data mount)
Do **not** run this cell if using Option B below.

In [ ]:
# ============================================================
# OPTION A: Load raw data from /data/ (do NOT run if using Option B)
# ============================================================
from lamf_analysis.code_ocean import capsule_data_utils as cdu
from lamf_analysis.code_ocean import docdb_utils
from aind_ophys_utils import dff as dff_utils

data_dir = Path('/root/capsule/data')
subject_ids = [755252, 767018, 767022, 783551, 785054, 782149, 788406, 790322,
               800792, 800995, 804363, 804670]


def get_attach_df(subject_id):
    session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
    processed_infos = (docdb_utils.get_processed_data_info(subject_id)
                       .sort_values('long_window')
                       .drop_duplicates(subset=['raw_name']))
    merged_df = processed_infos.merge(session_infos, left_on='raw_name',
                                      right_on='raw_asset_name', how='left')
    sczdrift_df = docdb_utils.get_derived_data_assets(subject_id, 'single-cell-zdrift-qc')
    sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                                'derived_asset_id': 'single_cell_zdrift_derived_asset_id'},
                       inplace=True)
    merged_df = merged_df.merge(
        sczdrift_df[['raw_name', 'single_cell_zdrift_derived_name',
                     'single_cell_zdrift_derived_asset_id']],
        on='raw_name', how='inner')
    return merged_df


def get_initial_f_results(row):
    processed_dir = data_dir / row.processed_name
    sczdrift_path = data_dir / row.single_cell_zdrift_derived_name
    plane_ids = cdu.get_plane_ids_from_processed_path(processed_dir)

    F_all, dff_short_all, dff_long_all = [], [], []
    baseline_short_all, baseline_long_all = [], []
    sczdrift_list = []

    for plane_id in plane_ids:
        plane_path = processed_dir / plane_id
        plane_depth = cdu.get_intended_depth(plane_path)
        roi_table = cdu.get_roi_table_from_plane_path(plane_path)
        valid_roi_inds = roi_table.query('valid_roi').cell_roi_id.values

        sczdrift_fn = next((sczdrift_path / plane_id).glob('*roi_time_profile.csv'))
        sczdrift_df = pd.read_csv(sczdrift_fn)
        zdrift_um = ((sczdrift_df.matched_plane_index_smoothed.max() -
                      sczdrift_df.matched_plane_index_smoothed.min()) * 0.75)
        max_frame_num = sczdrift_df.smoothed_frame_index.max()
        sczdrift_df = sczdrift_df.query('smoothed_frame_index == @max_frame_num')[
            ['cell_roi_id', 'fractional_change_from_first_frame']]
        sczdrift_df['plane_id'] = plane_id
        sczdrift_df['intended_depth'] = plane_depth
        sczdrift_df['z_drift_um'] = zdrift_um
        sczdrift_list.append(sczdrift_df)

        F = cdu.load_corrected_fluorescence(plane_path=plane_path)
        F_valid = F[valid_roi_inds, :]
        frame_rate = cdu.get_frame_rate_from_plane_path(plane_path)
        dff_short = cdu.load_dff_from_plane_path(plane_path)
        baseline_short = cdu.get_baseline_traces(plane_path)
        dff_long, baseline_long, _ = dff_utils.dff(F_valid, long_window=60*30, fs=frame_rate)

        F_all.append(F_valid)
        dff_short_all.append(dff_short[valid_roi_inds, :])
        dff_long_all.append(dff_long)
        baseline_short_all.append(baseline_short[valid_roi_inds, :])
        baseline_long_all.append(baseline_long)

    F_arr = np.concatenate(F_all, axis=0)
    baseline_short_arr = np.concatenate(baseline_short_all, axis=0)
    baseline_long_arr = np.concatenate(baseline_long_all, axis=0)
    dff_short_arr = np.concatenate(dff_short_all, axis=0)
    dff_long_arr = np.concatenate(dff_long_all, axis=0)
    sczdrift_all = pd.concat(sczdrift_list, ignore_index=True)

    F_noise_arr = noise_std(F_arr, 'mad')
    F_signal_arr = np.percentile(F_arr - baseline_short_arr, 99, axis=1)
    F_snr_arr = F_signal_arr / F_noise_arr
    F_skew_arr = skew(F_arr, axis=1)

    return (F_arr, dff_short_arr, dff_long_arr, baseline_short_arr, baseline_long_arr,
            sczdrift_all, F_noise_arr, F_signal_arr, F_snr_arr, F_skew_arr, frame_rate)


subject_id = subject_ids[0]
merged_df = get_attach_df(subject_id).sort_values('acquisition_date').reset_index(drop=True)

for _, row in tqdm(merged_df.iterrows(), total=len(merged_df)):
    session_key = row.session_key
    save_dir = save_dir_base / session_key

    if (save_dir / 'F0trend_all.npy').exists():
        print(f"{session_key}: already done, skipping")
        continue

    (F_all_array, dff_short_window_all_array, dff_long_window_all_array,
     baseline_short_window_all_array, baseline_long_window_all_array,
     sczdrift_df_all, F_noise, F_signal, F_snr, F_skewness, frame_rate) = \
        get_initial_f_results(row)

    bleaching_window = int(frame_rate * 60 * 5)
    baseline_diff = baseline_short_window_all_array - baseline_long_window_all_array
    zscored_baseline_diff = baseline_diff / np.std(baseline_diff, axis=1, keepdims=True)
    bleaching_metric = np.mean(zscored_baseline_diff[:, :bleaching_window], axis=1)
    sustained_metric = np.mean(zscored_baseline_diff[:, bleaching_window*2:], axis=1)

    b_inits = np.mean(F_all_array - baseline_long_window_all_array, axis=1)
    timestamps = np.arange(F_all_array.shape[1]) / frame_rate
    t_max = timestamps[-1]
    t_high_bound = t_max * 5

    def _fit_one_a(F, b_init):
        b_init = max(float(b_init), 1.0)
        F0trend, res = nonlinear_fit(
            F, timestamps, model,
            [F.mean(), b_init, b_init, b_init, t_max/2, 60, t_max/2],
            bounds=[
                (0, None), (0, None), (0, None), (0, None),
                (300, t_high_bound), (1, 300), (300, t_high_bound),
            ],
            M=M,
            fixed_sigma=None,
            backend='jax', dtype=jnp.float32)
        loss = float(np.mean(M.rho((F - F0trend) / res.sigma)))
        return F0trend, res, loss

    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(_fit_one_a)(F, b_inits[i])
        for i, F in enumerate(F_all_array)
    )
    F0trend_all, res_all, loss_all = map(list, zip(*results))

    save_dir.mkdir(parents=True, exist_ok=True)
    np.save(save_dir / 'F0trend_all.npy', np.array(F0trend_all))
    np.save(save_dir / 'res_all.npy', np.array(res_all, dtype=object))
    np.save(save_dir / 'loss_all.npy', np.array(loss_all))
    np.save(save_dir / 'F_all_array.npy', F_all_array)
    np.save(save_dir / 'F_noise.npy', F_noise)
    np.save(save_dir / 'F_signal.npy', F_signal)
    np.save(save_dir / 'F_snr.npy', F_snr)
    np.save(save_dir / 'F_skewness.npy', F_skewness)
    np.save(save_dir / 'timestamps.npy', timestamps)
    np.save(save_dir / 'baseline_short_window_all_array.npy', baseline_short_window_all_array)
    np.save(save_dir / 'baseline_long_window_all_array.npy', baseline_long_window_all_array)
    np.save(save_dir / 'dff_short_window_all_array.npy', dff_short_window_all_array)
    np.save(save_dir / 'dff_long_window_all_array.npy', dff_long_window_all_array)
    np.save(save_dir / 'bleaching_metric.npy', bleaching_metric)
    np.save(save_dir / 'sustained_metric.npy', sustained_metric)
    sczdrift_df_all.to_csv(save_dir / 'sczdrift_df_all.csv', index=False)
    # per-plane projection images and ROI tables
    plane_ids = cdu.get_plane_ids_from_processed_path(data_dir / row.processed_name)
    for plane_id in plane_ids:
        plane_path = data_dir / row.processed_name / plane_id
        mean_image = cdu.load_projection_image(plane_path, 'mean')
        max_image = cdu.load_projection_image(plane_path, 'max')
        roi_table = cdu.get_roi_table_from_plane_path(plane_path)
        np.save(save_dir / f"{plane_id}_mean_img.npy", mean_image)
        np.save(save_dir / f"{plane_id}_max_img.npy", max_image)
        roi_table.to_pickle(save_dir / f"{plane_id}_roi_table.pkl")
    print(f"{session_key}: done ({F_all_array.shape[0]} ROIs)")

---
## Option B — Load from `/scratch/first_try/` (precomputed arrays) — **RUN THIS**

Reads `F_all_array`, baselines, dff, and metadata from `first_try`.  
Only recomputes `F0trend` with the fixed IRLS (auto-sigma).  
All files are saved to `trend_only_default` for self-contained downstream use.

In [4]:
# ============================================================
# OPTION B: Load precomputed data from /scratch/first_try/
# ============================================================
first_try_dir = Path('/root/capsule/scratch/first_try')
session_dirs = sorted(first_try_dir.iterdir())
print(f"Found {len(session_dirs)} sessions in first_try")

for session_dir in tqdm(session_dirs):
    session_key = session_dir.name
    save_dir = save_dir_base / session_key

    if (save_dir / 'F0trend_all.npy').exists():
        print(f"{session_key}: already done, skipping")
        continue

    # --- Load precomputed arrays ---
    F_all_array = np.load(session_dir / 'F_all_array.npy')
    F_noise = np.load(session_dir / 'F_noise.npy')
    F_signal = np.load(session_dir / 'F_signal.npy')
    F_snr = np.load(session_dir / 'F_snr.npy')
    F_skewness = np.load(session_dir / 'F_skewness.npy')
    timestamps = np.load(session_dir / 'timestamps.npy')
    baseline_long = np.load(session_dir / 'baseline_long_window_all_array.npy')
    baseline_short = np.load(session_dir / 'baseline_short_window_all_array.npy')
    dff_long = np.load(session_dir / 'dff_long_window_all_array.npy')
    dff_short = np.load(session_dir / 'dff_short_window_all_array.npy')
    bleaching_metric = np.load(session_dir / 'bleaching_metric.npy')
    sustained_metric = np.load(session_dir / 'sustained_metric.npy')
    sczdrift_df = pd.read_csv(session_dir / 'sczdrift_df_all.csv')

    # --- Compute initial parameters for nonlinear_fit ---
    b_inits = np.mean(F_all_array - baseline_long, axis=1)
    t_max = timestamps[-1]
    t_high_bound = t_max * 5

    # --- Fit F0trend per ROI (trend only, no LOWESS, auto-sigma) ---
    def _fit_one(F, b_init):
        b_init = max(float(b_init), 1.0)
        F0trend, res = nonlinear_fit(
            F, timestamps, model,
            [F.mean(), b_init, b_init, b_init, t_max / 2, 60, t_max / 2],
            bounds=[
                (0, None),            # b_inf
                (0, None),            # b_slow
                (0, None),            # b_fast
                (0, None),            # b_bright
                (300, t_high_bound),  # t_slow
                (1, 300),             # t_fast
                (300, t_high_bound),  # t_bright
            ],
            M=M,
            fixed_sigma=None,  # key fix: auto-compute sigma from MAD(residuals)
            backend='jax',
            dtype=jnp.float32,
        )
        loss = float(np.mean(M.rho((F - F0trend) / res.sigma)))
        return F0trend, res, loss

    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(_fit_one)(F, b_inits[i])
        for i, F in enumerate(F_all_array)
    )
    F0trend_all, res_all, loss_all = map(list, zip(*results))

    # --- Save results ---
    save_dir.mkdir(parents=True, exist_ok=True)
    # new files
    np.save(save_dir / 'F0trend_all.npy', np.array(F0trend_all))
    np.save(save_dir / 'res_all.npy', np.array(res_all, dtype=object))
    np.save(save_dir / 'loss_all.npy', np.array(loss_all))
    # base arrays (copied for self-contained downstream access)
    np.save(save_dir / 'F_all_array.npy', F_all_array)
    np.save(save_dir / 'F_noise.npy', F_noise)
    np.save(save_dir / 'F_signal.npy', F_signal)
    np.save(save_dir / 'F_snr.npy', F_snr)
    np.save(save_dir / 'F_skewness.npy', F_skewness)
    np.save(save_dir / 'timestamps.npy', timestamps)
    np.save(save_dir / 'baseline_short_window_all_array.npy', baseline_short)
    np.save(save_dir / 'baseline_long_window_all_array.npy', baseline_long)
    np.save(save_dir / 'dff_short_window_all_array.npy', dff_short)
    np.save(save_dir / 'dff_long_window_all_array.npy', dff_long)
    np.save(save_dir / 'bleaching_metric.npy', bleaching_metric)
    np.save(save_dir / 'sustained_metric.npy', sustained_metric)
    sczdrift_df.to_csv(save_dir / 'sczdrift_df_all.csv', index=False)
    # per-plane projection images and ROI tables (copied from first_try)
    for src in sorted(session_dir.glob('*_mean_img.npy')):
        shutil.copy2(src, save_dir / src.name)
    for src in sorted(session_dir.glob('*_max_img.npy')):
        shutil.copy2(src, save_dir / src.name)
    for src in sorted(session_dir.glob('*_roi_table.pkl')):
        shutil.copy2(src, save_dir / src.name)

    n_bad = int(np.sum(np.array(F0trend_all).min(axis=1) < 0))
    print(f"{session_key}: done ({F_all_array.shape[0]} ROIs, {n_bad} with F0trend<0)")

Found 26 sessions in first_try


  0%|          | 0/26 [00:00<?, ?it/s]

755252_2024-11-12: done (508 ROIs, 0 with F0trend<0)
755252_2024-11-13: done (500 ROIs, 0 with F0trend<0)
755252_2024-11-14: done (492 ROIs, 0 with F0trend<0)
755252_2024-11-15: done (502 ROIs, 1 with F0trend<0)
755252_2024-11-18: done (511 ROIs, 0 with F0trend<0)
755252_2024-11-19: done (482 ROIs, 0 with F0trend<0)


/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


755252_2024-11-21: done (522 ROIs, 0 with F0trend<0)
755252_2024-11-22: done (474 ROIs, 2 with F0trend<0)
755252_2024-12-03: done (508 ROIs, 0 with F0trend<0)
755252_2024-12-04: done (487 ROIs, 0 with F0trend<0)
755252_2024-12-05: done (554 ROIs, 0 with F0trend<0)
755252_2024-12-06: done (517 ROIs, 0 with F0trend<0)


...

755252_2025-01-08: done (447 ROIs, 0 with F0trend<0)


/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


755252_2025-01-13: done (496 ROIs, 0 with F0trend<0)


/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


755252_2025-01-14: done (484 ROIs, 0 with F0trend<0)
